In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import warnings
warnings.filterwarnings('ignore')

%pip install kagglehub catboost xgboost tqdm -q

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

q1 = os.path.join(path, 'Q1_data.csv')
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(q1)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10,6))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequencey')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns='Order_ID')

In [ ]:
df.head()
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
# Task 2: Write your code here:
#print(df.isnull().sum())
#df = df.dropna()
numerical = df.select_dtypes(include=['number']).columns

for col in numerical:
  df[col] = df[col].fillna(df[col].median())

In [ ]:
categorical = df.select_dtypes(include=['object']).columns

for col in categorical:
  df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df.shape

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.shape

df.head()
df['Vehicle_Type'].value_counts()

In [ ]:
# Task 4: Write your code here:
#categorical_cols = df.select_dtypes(include=['object'])
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

le = LabelEncoder()
for col in categorical_cols:
  df[col] = le.fit_transform(df[col])

df.head()



In [ ]:
df.describe()

In [ ]:
scaler = StandardScaler()
X = df.drop(columns='Delivery_Time')
y = df['Delivery_Time']

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
X = df.drop(columns='Delivery_Time')
y = df['Delivery_Time']


X = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:
df['Delivery_Time'].value_counts()
'Its not a classification task, so we dont care about it being balanced or not'

In [ ]:
# Task 1: Write your code here:
'Already did that'
#X = df.drop(columns='Delivery_Time')
#y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=40, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X):
    X_train, X_test = X[train_idx], X[val_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae_scores.append(mean_absolute_error(y_true=y_test, y_pred=y_pred))

print(f"MAE:  ${np.mean(mae_scores):,.2f}")
feature_cols = ['Distance_km','Weather','Traffic_Level', 'Time_of_Day', 'Vehicle_Type' , 'Preparation_Time_min', 'Courier_Experience_yrs']


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)



plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_pred)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:

models = {
    'RandomForestRegressor': RandomForestRegressor(n_estimators=200, max_depth=20),
    'CatBoostRegressor' : CatBoostRegressor(verbose=0)
}
all_results = {}

for name in models:
  all_results[name] = {'mae':[]}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    all_results[model_name]["mae"].append(mae)






In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)

baseline_mse

"OH HELL NAHHHH"